# Mainz 3D: build a planning viewer in five steps

This notebook uses **basemap3d**, an unofficial, city-independent Python wrapper for basemap.de. Its Python interface follows familiar Folium conventions: create a `Map`, add layers with `.add_to(m)`, display `m`, and export with `m.save(...)`. CesiumJS renders the 3D scene in the browser. This is an independent teaching package, not Folium or an official Folium extension.

**Setup:** follow the English installation instructions in `README.md`, open this notebook from `notebooks/`, and run the cells in order. The setup cell switches to the repository root so bundled data paths work. All case-study instructions and scene labels below are German.

Each task contains at most seven lines of Python and saves its own HTML result in `results/`. No coordinates or long JavaScript blocks are embedded in the teaching cells. Later tasks use copies, so earlier results remain unchanged.

**Coordinate convention:** `Map(location=[latitude, longitude])` follows Folium; GeoJSON files use `[longitude, latitude]` in WGS84. The `zoom_start` setting is an approximate camera-distance preset, not an exact Leaflet zoom.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/Basemap3D/blob/main/notebooks/Mainz_3D_5_Tasks.ipynb)

**Colab:** click the badge, sign in if required, and run the setup cell below followed by the other cells. It fetches the public repository and installs the package automatically. No local Python installation or GPU runtime is needed. On a local Jupyter installation this setup cell switches to the repository root when needed.
**Colab:** Eine verbundene Laufzeit ist erforderlich. `m` bzw. `p2` am Zellenende (alternativ `.show()`) zeigt die Karte über den automatisch verwalteten Colab-Webserver an. Go Live und manuelle Serverbefehle sind nicht nötig. Nach einem Update eine neue Laufzeit verwenden und alle Zellen erneut ausführen.


In [ ]:
import os, subprocess, sys
if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    target = "/content/basemap3d-course"
    if not os.path.isdir(target + "/.git"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yfeng-hsm/Basemap3D.git", target], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    sys.path.insert(0, target)
elif os.path.isdir("../basemap3d"):
    os.chdir("..")


In [ ]:
from basemap3d import Map, WmsTileLayer, GeoJson, NearbyBuildings, SunShadows

## Task 1 — Basemap and 3D city

Add a basemap, official LoD2 buildings, and the matching terrain.

### Fallbeispiel Mainz: Stadtmodell am Schloss

**Lernziel:** Unterscheiden Sie die zweidimensionale Basiskarte, das Gelände und die dreidimensionalen Gebäude. Ändern Sie `zoom_start`, `heading` oder `pitch` und vergleichen Sie die Ansichten.

**Ergebnis:** Eine navigierbare 3D-Stadtansicht, noch ohne Luftbild, Parkplatzflächen, räumliche Abfrage oder Schattensteuerung. Ausgabe: `results/task1_3d.html`.

In [ ]:
m1 = Map(location=[50.0063, 8.2692], zoom_start=16,
         title="Mainz · Stadtmodell am Schloss")
m1.save("results/task1_3d.html")
m1

## Task 2 — Add a raster layer

Add a WMS raster overlay and inspect its opacity.

### Fallbeispiel Mainz: Luftbild als Rasterebene

**Lernziel:** Ergänzen Sie das amtliche DOP20-Luftbild aus Rheinland-Pfalz. Es handelt sich um ein Orthophoto mit 20 cm Bodenauflösung, nicht um ein Satellitenbild. Es erzeugt keine Fassadentexturen für das Stadtmodell.

**Aufgabe:** Vergleichen Sie `opacity=0.8` mit `opacity=0.4`, oder verwenden Sie den Regler „Luftbild-Deckkraft“. Welche Details eines Parkplatzes erkennen Sie im Luftbild besser?

**Ergebnis:** Stadtmodell mit schaltbarer Rasterebene und Deckkraftregler. Ausgabe: `results/task2_raster.html`.

In [ ]:
m2 = m1.copy()
WmsTileLayer(
    url="https://geo4.service24.rlp.de/wms/rp_dop20.fcgi",
    layers="rp_dop20", opacity=0.8,
).add_to(m2)
m2.save("results/task2_raster.html")
m2

## Task 3 — Add a vector layer

Read a local GeoJSON file and overlay polygon features.

### Fallbeispiel Mainz: Fünf Parkplatzflächen

**Lernziel:** Stellen Sie die fünf Flächen Schloss, Kaisertor, Bruchwegstadion (Südseite), Dalheimer Weg und Volkspark (Westseite) dar. Die Koordinaten und Attribute werden aus `data/parking.geojson` gelesen.

**Aufgabe:** Ändern Sie `fill_opacity`. Wählen Sie einen Namen in der Legende und untersuchen Sie die Lage der Fläche. Noch ist keine räumliche Abfrage oder Popup-Steuerung aktiviert.

**Ergebnis:** Fünf farbige Polygonflächen mit Legende und Navigation. Ausgabe: `results/task3_vector.html`.

In [ ]:
m3 = m2.copy()
GeoJson("data/parking.geojson", fill_opacity=0.25).add_to(m3)
m3.save("results/task3_vector.html")
m3

## Task 4 — Spatial query and popup

Select nearby building footprints and enable attribute popups.

### Fallbeispiel Mainz: Gebäude im Umfeld

**Lernziel:** Wählen Sie Gebäudeumrisse aus, deren kürzester Abstand zu mindestens einer Parkplatzfläche höchstens 75 m beträgt. Umrisse, die eine Fläche schneiden oder in ihr liegen, haben Abstand 0. Ein Umriss wird nur einmal gezählt. Die Abfrage gilt für alle fünf Flächen gleichzeitig.

**Aufgabe:** Vergleichen Sie 75 m mit 150 m. Die Trefferzahl darf beim Vergrößern des Suchabstands nicht abnehmen. Klicken Sie auf einen Parkplatz oder seinen Namen: Das Popup zeigt Name, OSM-ID, Zugang und Beschreibung. Blenden Sie Gebäude aus, wenn sie eine Fläche verdecken.

**Ergebnis:** Goldene Hervorhebung, Trefferzahl, Abstandsregler und Attribut-Popup. Ausgabe: `results/task4_query_popup.html`.

Die Gebäudeumrisse werden automatisch aus OpenStreetMap für die aktuellen Untersuchungsflächen geladen. Wenn Sie die Polygondatei durch ein anderes Gebiet ersetzen, werden die zugehörigen Gebäude neu abgefragt. Die lokale Meter-Näherung wird aus den Koordinaten des jeweiligen Gebiets abgeleitet. Das LoD2-Modell stammt aus einer anderen Quelle. Ein 3-m-Rand verbessert die visuelle Färbung, verändert aber weder Auswahlabstand noch Trefferzahl. Unterschiede zwischen den Quellen bleiben möglich; die Trefferzahl ist keine amtliche Gebäudestatistik.

In [ ]:
m4 = m3.copy()
NearbyBuildings(distance_m=75, popup=True).add_to(m4)
m4.save("results/task4_query_popup.html")
m4

## Task 5 — Sun-based shadows

Add date and local-time controls for the sun and model shadows.

### Fallbeispiel Mainz: Tageszeit und Verschattung

**Lernziel:** Beobachten Sie, wie sich Richtung und Länge der Gebäudeschatten mit Datum und Uhrzeit ändern. Die Zeitsteuerung verwendet `Europe/Berlin`, einschließlich Sommerzeit.

**Aufgabe:** Vergleichen Sie 09:00, 13:00 und 16:00. Wählen Sie danach einen Sommer- und einen Wintertag. Deaktivieren Sie bei Bedarf die goldene Hervorhebung und wechseln Sie zur Basiskarte, damit die Modellschatten besser erkennbar sind.

**Ergebnis:** Ein vollständiger Viewer mit allen bisherigen Ebenen sowie Datums- und Zeitsteuerung. Ausgabe: `results/task5_sun_shadows.html`.

Schatten, die bereits im Luftbild enthalten sind, bewegen sich nicht. Das vereinfachte LoD2-Modell ersetzt keine professionelle Verschattungs- oder thermische Analyse.

In [ ]:
m5 = m4.copy()
SunShadows(date="2026-10-25", hour=13.0).add_to(m5)
m5.save("results/task5_sun_shadows.html")
m5

## Submission and further reading

### Abgabe

Geben Sie das bearbeitete Notebook, fünf HTML-Dateien und je einen Screenshot aus Task 3 und Task 5 ab. Erläutern Sie kurz den Unterschied zwischen Raster- und Vektordaten, die Abfrageregel und den Einfluss der Uhrzeit auf den Schattenwurf. Behalten Sie alle Quellenangaben bei.

### Package notes

- Run the later cells again after changing an earlier map. `.copy()` creates independent snapshots.
- Run a final `m1`–`m5` expression to display a scene. `m.save(...)` exports the same scene. HTML results embed their configuration and local vectors; online imagery, terrain and 3D models still require internet access.
- Trust your own notebook before running JavaScript output. GitHub's notebook preview does not execute the 3D viewer; use the HTML gallery or run Jupyter locally.
- `NearbyBuildings()` derives its requests from the current polygons, with a 24-hour local OSM response cache. It does not use the five-site snapshot by default. For offline teaching, explicitly pass `"data/mainz_buildings_snapshot.geojson"`; that optional snapshot covers only the original case-study area. The lightweight downloader limits request count; provide a local file for large areas.
- Package documentation is in English: [README](README.md), [API](docs/API.md), [data and limitations](docs/DATA.md), [teaching guide](docs/TEACHING.md). German examples are in [Fallbeispiele](docs/beispiele.html).

Data: basemap.de / GeoBasis-DE/BKG (online LoD2 and DGM5), LVermGeoRP DOP20, and © OpenStreetMap contributors (ODbL; retrieved 2026-10-25). P3/P5 use descriptive local names. P4 is a restricted campus parking area. See [data documentation](docs/DATA.md) for attribution and licensing. Retrieval year does not imply a common acquisition year.